### Imports

In [1]:
import sys
sys.dont_write_bytecode = True


import torch
import numpy as np
import random
import os

def set_seeds(seed_value=42):
    """Sets seeds for reproducibility."""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)

set_seeds(42) 

import json
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import warnings
import logging
from datetime import datetime
warnings.filterwarnings('ignore')

from model import get_model
from config import CFG
from dataset import get_dataset_class
from transform import get_transforms
from runner import run_baseline, run_lodo

torch.manual_seed(CFG["system"]["seed"])
np.random.seed(CFG["system"]["seed"])

device = CFG["system"]["device"]
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

DS = "VLCS"
MODEL_NAME = "resnet18"

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu126 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1126 04:30:16.280000 35592 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Device: cuda
PyTorch: 2.8.0+cu126


### DataLoading

In [2]:
train_transform, test_transform = get_transforms(img_size=224, augment=False, use_imagenet_norm=False)

DatasetClass = get_dataset_class(DS)

ld = DatasetClass(
    data_root=CFG["datasets"][DS]["root"],
    transform=train_transform,
    batch_size=CFG["train"]["batch_size"]
)

print("\nData loaders ready!")


Data loaders ready!


### Logging

In [3]:
dataset_name = DS
base_dir = os.path.join(os.getcwd(), dataset_name)
subdirs = ["logs", "checkpoints", "plots"]

for sub in subdirs:
    os.makedirs(os.path.join(base_dir, sub), exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_file = os.path.join(base_dir, "logs", f"train_{timestamp}.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(f"{dataset_name}_logger")

logger.info(f"Initialized experiment directories for {dataset_name}")
logger.info(f"Logs: {os.path.join(base_dir, 'logs')}")
logger.info(f"Checkpoints: {os.path.join(base_dir, 'checkpoints')}")
logger.info(f"Plots: {os.path.join(base_dir, 'plots')}")

2025-11-26 04:30:17,671 | INFO | Initialized experiment directories for VLCS
2025-11-26 04:30:17,672 | INFO | Logs: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\VLCS\logs
2025-11-26 04:30:17,672 | INFO | Checkpoints: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\VLCS\checkpoints
2025-11-26 04:30:17,673 | INFO | Plots: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\VLCS\plots


### Setup

In [4]:
domains = CFG["datasets"][DS]["domains"]
loaders = {d: {"train": ld.get_dataloader(d, train=True), "val": ld.get_dataloader(d, train=False)} for d in domains}
ckpt_root = os.path.join(base_dir, "checkpoints")
log_dir = os.path.join(base_dir, "logs")
plots_dir = os.path.join(base_dir, "plots")
os.makedirs(ckpt_root, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
model_factory = lambda cfg, dataset_key: get_model(cfg,dataset=DS)
optimizer_fn = lambda model: optim.AdamW(model.parameters(), lr=CFG["train"]["lr"], weight_decay=CFG["train"].get("weight_decay", 0.01))
device = CFG["system"]["device"]
epochs = CFG["train"]["epochs"]


{
  "lodo_results": {
    "art_painting": 0.8341463414634146,
    "cartoon": 0.7974413646055437,
    "photo": 0.9580838323353293,
    "sketch": 0.6017811704834606
  },
  "timestamp": "20251004_020611"
}

### Leave One Domain Out

In [5]:
lodo_results, lodo_mean, lodo_summary = run_lodo(
    model_fn=model_factory,
    CFG=CFG,
    logger=logger,
    dataset_key=DS,
    domains=domains,
    loaders=loaders,
    optimizer_fn=optimizer_fn,
    device=device,
    ckpt_root=ckpt_root,
    log_dir=log_dir,
    epochs=epochs
)

2025-11-26 04:30:17,998 | INFO | === LODO: Leaving out domain 'VOC2007' ===



=== LODO: Leaving out domain 'VOC2007' ===


Evaluating: 100%|██████████| 27/27 [00:08<00:00,  3.19it/s]
2025-11-26 04:30:53,439 | INFO | [VOC2007] Epoch 1/10 | Train - Loss: 0.6676, Cls: 0.6587, GRQO: 0.0090, Acc: 0.7487 | Val - Loss: 0.7131, Cls: 0.7117, GRQO: 0.0014, Acc: 0.7473
2025-11-26 04:30:53,539 | INFO | [VOC2007] New best val acc: 0.7473


[VOC2007] Epoch 1/10 | Train - Loss: 0.6676, Cls: 0.6587, GRQO: 0.0090, Acc: 0.7487 | Val - Loss: 0.7131, Cls: 0.7117, GRQO: 0.0014, Acc: 0.7473
[VOC2007] New best val acc: 0.7473


Evaluating: 100%|██████████| 27/27 [00:08<00:00,  3.24it/s]
2025-11-26 04:31:29,723 | INFO | [VOC2007] Epoch 2/10 | Train - Loss: 0.2965, Cls: 0.2946, GRQO: 0.0018, Acc: 0.8973 | Val - Loss: 0.7347, Cls: 0.7341, GRQO: 0.0007, Acc: 0.7488
2025-11-26 04:31:29,805 | INFO | [VOC2007] New best val acc: 0.7488


...

[SUN09] Epoch 8/10 | Train - Loss: 0.0417, Cls: 0.0419, GRQO: -0.0002, Acc: 0.9863 | Val - Loss: 1.6066, Cls: 1.6058, GRQO: 0.0008, Acc: 0.6807


Evaluating: 100%|██████████| 26/26 [00:13<00:00,  1.86it/s]
2025-11-26 05:01:51,672 | INFO | [SUN09] Epoch 9/10 | Train - Loss: 0.0327, Cls: 0.0334, GRQO: -0.0007, Acc: 0.9887 | Val - Loss: 1.2882, Cls: 1.2881, GRQO: 0.0001, Acc: 0.7255


[SUN09] Epoch 9/10 | Train - Loss: 0.0327, Cls: 0.0334, GRQO: -0.0007, Acc: 0.9887 | Val - Loss: 1.2882, Cls: 1.2881, GRQO: 0.0001, Acc: 0.7255


Evaluating: 100%|██████████| 26/26 [00:13<00:00,  1.90it/s]
2025-11-26 05:02:32,278 | INFO | [SUN09] Epoch 10/10 | Train - Loss: 0.0177, Cls: 0.0187, GRQO: -0.0010, Acc: 0.9941 | Val - Loss: 1.5640, Cls: 1.5639, GRQO: 0.0001, Acc: 0.6968
2025-11-26 05:02:32,279 | INFO | [SUN09] Best Acc: 0.7450
2025-11-26 05:02:32,279 | INFO | ------------------------------------------------------------
2025-11-26 05:02:32,281 | INFO | LODO finished | Mean Acc: 0.7986 | Summary saved to d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\VLCS\logs\lodo_summary_20251126_050232.json


[SUN09] Epoch 10/10 | Train - Loss: 0.0177, Cls: 0.0187, GRQO: -0.0010, Acc: 0.9941 | Val - Loss: 1.5640, Cls: 1.5639, GRQO: 0.0001, Acc: 0.6968
[SUN09] Best Acc: 0.7450
------------------------------------------------------------
LODO finished | Mean Acc: 0.7986
Summary saved to d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\VLCS\logs\lodo_summary_20251126_050232.json


### Baseline

In [6]:
baseline_results, baseline_mean = run_baseline(
    model_name=MODEL_NAME,
    CFG=CFG,
    logger=logger,
    dataset_key=DS,
    domains=domains,
    loaders=loaders,
    optimizer_fn=optimizer_fn,
    device=device,
    epochs=CFG["train"]["epochs"]
)

2025-11-26 05:02:32,287 | INFO | Initializing ResNet baseline: resnet18
2025-11-26 05:02:32,361 | INFO | === Baseline LODO: Leaving out domain 'VOC2007' ===


Initializing ResNet baseline: resnet18

=== Baseline LODO: Leaving out domain 'VOC2007' ===


2025-11-26 05:03:07,940 | INFO | [VOC2007] Epoch 1/10 | Train - Loss: 0.6276, Acc: 0.7650 | Val Acc: 0.7556


[VOC2007] Epoch 1/10 | Train - Loss: 0.6276, Acc: 0.7650 | Val Acc: 0.7556


2025-11-26 05:03:43,736 | INFO | [VOC2007] Epoch 2/10 | Train - Loss: 0.2770, Acc: 0.9060 | Val Acc: 0.7310


...

[SUN09] Epoch 8/10 | Train - Loss: 0.0030, Acc: 1.0000 | Val Acc: 0.7313


2025-11-26 05:33:19,869 | INFO | [SUN09] Epoch 9/10 | Train - Loss: 0.0020, Acc: 1.0000 | Val Acc: 0.7313


[SUN09] Epoch 9/10 | Train - Loss: 0.0020, Acc: 1.0000 | Val Acc: 0.7313


2025-11-26 05:34:00,724 | INFO | [SUN09] Epoch 10/10 | Train - Loss: 0.0017, Acc: 1.0000 | Val Acc: 0.7297
2025-11-26 05:34:00,724 | INFO | [SUN09] Best Val Acc: 0.7346
2025-11-26 05:34:00,734 | INFO | ------------------------------------------------------------
2025-11-26 05:34:00,736 | INFO | Baseline LODO (resnet18) finished | Mean Acc: 0.7870


[SUN09] Epoch 10/10 | Train - Loss: 0.0017, Acc: 1.0000 | Val Acc: 0.7297
[SUN09] Best Val Acc: 0.7346
------------------------------------------------------------
Baseline LODO (resnet18) finished | Mean Acc: 0.7870
